In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("test") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/07 20:05:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# using nyc taxi data / 2021 / january / higher volume trip data 
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2021-01.parquet

--2026-03-07 20:08:33--  https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2021-01.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 99.84.245.193, 99.84.245.141, 99.84.245.157, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|99.84.245.193|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 308924937 (295M) [application/x-www-form-urlencoded]
Saving to: ‘fhvhv_tripdata_2021-01.parquet’

fhvhv_tripdata_2021 100%[===================>] 294.61M  47.2MB/s    in 6.3s    

2026-03-07 20:08:39 (46.5 MB/s) - ‘fhvhv_tripdata_2021-01.parquet’ saved [308924937/308924937]



In [4]:
!wc -l fhvhv_tripdata_2021-01.parquet

1006794 fhvhv_tripdata_2021-01.parquet


In [8]:
df = spark.read.parquet("fhvhv_tripdata_2021-01.parquet")
    # .option("header", "true") \
df.show()

+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+----+----------+-------------------+-----------------+------------------+----------------+--------------+
|hvfhs_license_num|dispatching_base_num|originating_base_num|   request_datetime|  on_scene_datetime|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|trip_miles|trip_time|base_passenger_fare|tolls| bcf|sales_tax|congestion_surcharge|airport_fee|tips|driver_pay|shared_request_flag|shared_match_flag|access_a_ride_flag|wav_request_flag|wav_match_flag|
+-----------------+--------------------+--------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+----------+---------+-------------------+-----+----+---------+--------------------+-----------+--

In [10]:
df

DataFrame[hvfhs_license_num: string, dispatching_base_num: string, originating_base_num: string, request_datetime: timestamp_ntz, on_scene_datetime: timestamp_ntz, pickup_datetime: timestamp_ntz, dropoff_datetime: timestamp_ntz, PULocationID: bigint, DOLocationID: bigint, trip_miles: double, trip_time: bigint, base_passenger_fare: double, tolls: double, bcf: double, sales_tax: double, congestion_surcharge: double, airport_fee: double, tips: double, driver_pay: double, shared_request_flag: string, shared_match_flag: string, access_a_ride_flag: string, wav_request_flag: string, wav_match_flag: string]

In [11]:
df.head(5)

[Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', originating_base_num='B02682', request_datetime=datetime.datetime(2021, 1, 1, 0, 28, 9), on_scene_datetime=datetime.datetime(2021, 1, 1, 0, 31, 42), pickup_datetime=datetime.datetime(2021, 1, 1, 0, 33, 44), dropoff_datetime=datetime.datetime(2021, 1, 1, 0, 49, 7), PULocationID=230, DOLocationID=166, trip_miles=5.26, trip_time=923, base_passenger_fare=22.28, tolls=0.0, bcf=0.67, sales_tax=1.98, congestion_surcharge=2.75, airport_fee=None, tips=0.0, driver_pay=14.99, shared_request_flag='N', shared_match_flag='N', access_a_ride_flag=' ', wav_request_flag='N', wav_match_flag='N'),
 Row(hvfhs_license_num='HV0003', dispatching_base_num='B02682', originating_base_num='B02682', request_datetime=datetime.datetime(2021, 1, 1, 0, 45, 56), on_scene_datetime=datetime.datetime(2021, 1, 1, 0, 55, 19), pickup_datetime=datetime.datetime(2021, 1, 1, 0, 55, 19), dropoff_datetime=datetime.datetime(2021, 1, 1, 1, 18, 21), PULocationID=152, DO

In [12]:
df.schema

StructType([StructField('hvfhs_license_num', StringType(), True), StructField('dispatching_base_num', StringType(), True), StructField('originating_base_num', StringType(), True), StructField('request_datetime', TimestampNTZType(), True), StructField('on_scene_datetime', TimestampNTZType(), True), StructField('pickup_datetime', TimestampNTZType(), True), StructField('dropoff_datetime', TimestampNTZType(), True), StructField('PULocationID', LongType(), True), StructField('DOLocationID', LongType(), True), StructField('trip_miles', DoubleType(), True), StructField('trip_time', LongType(), True), StructField('base_passenger_fare', DoubleType(), True), StructField('tolls', DoubleType(), True), StructField('bcf', DoubleType(), True), StructField('sales_tax', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('airport_fee', DoubleType(), True), StructField('tips', DoubleType(), True), StructField('driver_pay', DoubleType(), True), StructField('shared_re

In [14]:
# if that was a csv and i had to define types myself: 
"""
from pyspark.sql import types 
from pyspark.sql import types

schema = types.StructType([
    types.StructField('hvfhs_license_num', types.StringType(), True),
    types.StructField('dispatching_base_num', types.StringType(), True),
    types.StructField('pickup_datetime', types.TimestampType(), True),
    types.StructField('dropoff_datetime', types.TimestampType(), True),
    types.StructField('PULocationID', types.IntegerType(), True),
    types.StructField('DOLocationID', types.IntegerType(), True),
    types.StructField('SR_Flag', types.StringType(), True)
])
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv("file.csv")
"""


'\nfrom pyspark.sql import types \nfrom pyspark.sql import types\n\nschema = types.StructType([\n    types.StructField(\'hvfhs_license_num\', types.StringType(), True),\n    types.StructField(\'dispatching_base_num\', types.StringType(), True),\n    types.StructField(\'pickup_datetime\', types.TimestampType(), True),\n    types.StructField(\'dropoff_datetime\', types.TimestampType(), True),\n    types.StructField(\'PULocationID\', types.IntegerType(), True),\n    types.StructField(\'DOLocationID\', types.IntegerType(), True),\n    types.StructField(\'SR_Flag\', types.StringType(), True)\n])\ndf = spark.read     .option("header", "true")     .schema(schema)     .csv("file.csv")\n'

In [36]:
# somehow we want to break it now into multiple files (partitions) so that instead of only one executor,
# many executors can work on it

df = df.repartition(24)  # this is lazy command: only when we save the df it will be partitioned


In [37]:
df.rdd.getNumPartitions()  # 24 requested partitions ↓ 16 actual partitions: 
# Adaptive Query Execution (AQE) to reduce small files 

[Stage 18:====================================================>   (15 + 1) / 16]

24

In [38]:
df.write.parquet("fhvhv/2021/01/", mode="overwrite")

26/03/07 22:50:01 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/03/07 22:50:01 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/03/07 22:50:01 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/03/07 22:50:01 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/03/07 22:50:01 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/03/07 22:50:01 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/03/07 22:50:01 WARN MemoryManager: Total allocation exceeds 95.

In [39]:
# df.explain()
df.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (7)
+- == Final Plan ==
   ResultQueryStage (5)
   +- ShuffleQueryStage (4), Statistics(sizeInBytes=2.9 GiB, rowCount=1.19E+7)
      +- Exchange (3)
         +- * ColumnarToRow (2)
            +- Scan parquet  (1)
+- == Initial Plan ==
   Exchange (6)
   +- Scan parquet  (1)


(1) Scan parquet 
Output [24]: [hvfhs_license_num#48, dispatching_base_num#49, originating_base_num#50, request_datetime#51, on_scene_datetime#52, pickup_datetime#53, dropoff_datetime#54, PULocationID#55L, DOLocationID#56L, trip_miles#57, trip_time#58L, base_passenger_fare#59, tolls#60, bcf#61, sales_tax#62, congestion_surcharge#63, airport_fee#64, tips#65, driver_pay#66, shared_request_flag#67, shared_match_flag#68, access_a_ride_flag#69, wav_request_flag#70, wav_match_flag#71]
Batched: true
Location: InMemoryFileIndex [file:/home/maria.szepek/Documents/DTCDE/github/data-engineering-zoomcamp/module6-batch/notebooks/fhvhv_tripdata_2021-01.parquet]
ReadSchema: struct<hvfhs_lic

In [ ]:
# next time: run top or htop to observe the active cpu threads 

In [ ]:
# apparently what happenend: memory pressure -> reduced row group sizes -> job finished 

In [28]:
# what i can do to allocate more memory: 
"""
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("test") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()
"""

None


In [42]:
# learning: the df had already 16 partitions before i repartitioned it! 
# he did that to maximize parallelization. he has a max partition bytes parameter, 
# and computes file size / max part. size = nr of partitions. 

spark.conf.get("spark.sql.files.maxPartitionBytes")

'134217728b'

In [ ]:
# compression csv to parquet it roughly factor 4! 

In [45]:
##################
# SPARK DATAFRAMES
##################

In [46]:
df = spark.read.parquet("fhvhv/2021/01/")

In [47]:
df.printSchema()

root
 |-- hvfhs_license_num: string (nullable = true)
 |-- dispatching_base_num: string (nullable = true)
 |-- originating_base_num: string (nullable = true)
 |-- request_datetime: timestamp_ntz (nullable = true)
 |-- on_scene_datetime: timestamp_ntz (nullable = true)
 |-- pickup_datetime: timestamp_ntz (nullable = true)
 |-- dropoff_datetime: timestamp_ntz (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- trip_miles: double (nullable = true)
 |-- trip_time: long (nullable = true)
 |-- base_passenger_fare: double (nullable = true)
 |-- tolls: double (nullable = true)
 |-- bcf: double (nullable = true)
 |-- sales_tax: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- tips: double (nullable = true)
 |-- driver_pay: double (nullable = true)
 |-- shared_request_flag: string (nullable = true)
 |-- shared_match_flag: string (nullable = true)
 |-- access_a_ride_f

In [ ]:
# thats one reason that parquet files are smaller! they know the data types, and then they can store 
# more efficiently

In [52]:
df.select ('pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID') \
    .filter(df.hvfhs_license_num == 'HV0003') 

DataFrame[pickup_datetime: timestamp_ntz, dropoff_datetime: timestamp_ntz, PULocationID: bigint, DOLocationID: bigint]

In [53]:
df.select ('pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID') \
    .filter(df.hvfhs_license_num == 'HV0003') \
    .head(5)    

[Row(pickup_datetime=datetime.datetime(2021, 1, 5, 12, 42, 9), dropoff_datetime=datetime.datetime(2021, 1, 5, 13, 4, 44), PULocationID=22, DOLocationID=261),
 Row(pickup_datetime=datetime.datetime(2021, 1, 2, 14, 16, 6), dropoff_datetime=datetime.datetime(2021, 1, 2, 14, 22, 35), PULocationID=189, DOLocationID=181),
 Row(pickup_datetime=datetime.datetime(2021, 1, 6, 8, 26, 45), dropoff_datetime=datetime.datetime(2021, 1, 6, 8, 53), PULocationID=56, DOLocationID=75),
 Row(pickup_datetime=datetime.datetime(2021, 1, 28, 1, 7, 33), dropoff_datetime=datetime.datetime(2021, 1, 28, 1, 17, 39), PULocationID=169, DOLocationID=69),
 Row(pickup_datetime=datetime.datetime(2021, 1, 17, 9, 0, 11), dropoff_datetime=datetime.datetime(2021, 1, 17, 9, 13, 5), PULocationID=169, DOLocationID=69)]

In [ ]:
# actions vs transformations 

# TRANSFORMATIONS: 
# selecting columns, filtering 

# ACTIONS:
# show 

In [ ]:
##### 
# THESE LOOKS MUCH LIKE SQL
# THEN WHY WE ARE USING PYSPARK? 
# IT HAS AMONG OTHERS, MORE FLEXIBILITY, FOR EXAMPLE UDF: USER DEFINED FUNCTIONS

In [55]:
# Functions that are already available: 
from pyspark.sql import functions as F

In [60]:
# F.to_date()
df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .select('pickup_date', 'dropoff_date', 'PULocationID', 'DOLocationID') \
    .show()
# .withColumn('dropoff_datetime', F.to_date(df.dropoff_datetime)) \ # would overwrite

+-----------+------------+------------+------------+
|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-----------+------------+------------+------------+
| 2021-01-05|  2021-01-05|          22|         261|
| 2021-01-02|  2021-01-02|         189|         181|
| 2021-01-06|  2021-01-06|          56|          75|
| 2021-01-28|  2021-01-28|         169|          69|
| 2021-01-17|  2021-01-17|         169|          69|
| 2021-01-29|  2021-01-29|         198|          37|
| 2021-01-14|  2021-01-14|         164|         145|
| 2021-01-10|  2021-01-10|          58|         213|
| 2021-01-21|  2021-01-21|          51|         174|
| 2021-01-12|  2021-01-12|         169|         250|
| 2021-01-06|  2021-01-06|          74|         152|
| 2021-01-16|  2021-01-16|         145|         138|
| 2021-01-22|  2021-01-22|          37|          80|
| 2021-01-01|  2021-01-01|         248|         213|
| 2021-01-19|  2021-01-19|          91|          14|
| 2021-01-05|  2021-01-05|         236|       

In [63]:
# we can also define our own functions especially in python where i can store in my own git repo and test
# (compared to data warehouse for example)

# function that does something that would not be super easy with sql
def crazy_stuff(base_num):
    num = int(base_num[1:])
    if num % 7 == 0:
        return f's/{num:03x}' # num but in hex
    elif num % 3 ==0:
        return f'a/{num:03x}'
    else:
        return f'e/{num:03x}'

# lets assume this is the business requirement for whatever reason.

# now this logic can live in its separate module and be covered by unit tests

crazy_stuff('B02884')

's/b44'

In [65]:
from pyspark.sql import types
crazy_stuff_udf = F.udf(crazy_stuff, returnType=types.StringType())

In [67]:
df \
    .withColumn('pickup_date', F.to_date(df.pickup_datetime)) \
    .withColumn('dropoff_date', F.to_date(df.dropoff_datetime)) \
    .withColumn('base_id', crazy_stuff_udf(df.dispatching_base_num)) \
    .select('base_id', 'pickup_date', 'dropoff_date', 'PULocationID', 'DOLocationID') \
    .show()


+-------+-----------+------------+------------+------------+
|base_id|pickup_date|dropoff_date|PULocationID|DOLocationID|
+-------+-----------+------------+------------+------------+
|  e/acc| 2021-01-05|  2021-01-05|          22|         261|
|  e/b33| 2021-01-02|  2021-01-02|         189|         181|
|  s/acd| 2021-01-06|  2021-01-06|          56|          75|
|  s/acd| 2021-01-28|  2021-01-28|         169|          69|
|  e/b3b| 2021-01-17|  2021-01-17|         169|          69|
|  e/9ce| 2021-01-29|  2021-01-29|         198|          37|
|  e/acc| 2021-01-14|  2021-01-14|         164|         145|
|  e/9ce| 2021-01-10|  2021-01-10|          58|         213|
|  e/9ce| 2021-01-21|  2021-01-21|          51|         174|
|  e/95b| 2021-01-12|  2021-01-12|         169|         250|
|  e/b47| 2021-01-06|  2021-01-06|          74|         152|
|  e/9ce| 2021-01-16|  2021-01-16|         145|         138|
|  e/b30| 2021-01-22|  2021-01-22|          37|          80|
|  s/b44| 2021-01-01|  2